# Hola, mundo en LangChain

## Instalar librerías principales y configuración de API Key de OpenAI

In [ ]:
%%capture
%pip install langchain pypdf openai chromadb tiktoken langchain-chroma langchain-huggingface sentence-transformers

In [9]:
from dotenv import load_dotenv
import os

load_dotenv(override=True) # Loads variables from .env file, overriding existing env vars

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
if not OPENAI_API_KEY:
    raise ValueError("No OPENAI_API_KEY found in .env file.")

## Carga de documents

In [11]:
import requests
from langchain_community.document_loaders import PyPDFLoader
import os

# Definir la ruta base del proyecto (un nivel arriba de notebooks/)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
resources_dir = os.path.join(project_root, 'resources')

# Crear la carpeta resources si no existe
if not os.path.exists(resources_dir):
    os.makedirs(resources_dir)
    print(f"Carpeta creada en: {resources_dir}")

urls = [
    'https://arxiv.org/pdf/2306.06031v1.pdf',
    'https://arxiv.org/pdf/2306.12156v1.pdf',
    'https://arxiv.org/pdf/2306.14289v1.pdf',
    'https://arxiv.org/pdf/2305.10973v1.pdf',
    'https://arxiv.org/pdf/2306.13643v1.pdf'
]

ml_papers = []

for i, url in enumerate(urls):
    filename = os.path.join(resources_dir, f'paper{i+1}.pdf')

    if os.path.exists(filename):
        print(f'Archivo existente, se reutiliza: {filename}')
    else:
        response = requests.get(url)
        response.raise_for_status()
        with open(filename, 'wb') as f:
            f.write(response.content)
        print(f'Descargado {filename}')

    loader = PyPDFLoader(filename)
    ml_papers.extend(loader.load())

# Utiliza la lista ml_papers para acceder a los elementos de todos los documentos descargados
print('Contenido de ml_papers:')
print()

Archivo existente, se reutiliza: c:\Users\julio_arismendi\git\langchain_course\resources\paper1.pdf
Archivo existente, se reutiliza: c:\Users\julio_arismendi\git\langchain_course\resources\paper2.pdf
Archivo existente, se reutiliza: c:\Users\julio_arismendi\git\langchain_course\resources\paper3.pdf
Archivo existente, se reutiliza: c:\Users\julio_arismendi\git\langchain_course\resources\paper4.pdf
Archivo existente, se reutiliza: c:\Users\julio_arismendi\git\langchain_course\resources\paper5.pdf
Contenido de ml_papers:



In [ ]:
type(ml_papers), len(ml_papers), ml_papers[3]

## Split de documents

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
    )

documents = text_splitter.split_documents(ml_papers)

In [ ]:
len(documents), documents[10]

## Embeddings e ingesta a base de datos vectorial

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import os

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

chroma_dir = os.path.join(resources_dir, "chroma_db")
vectorstore = Chroma(
    collection_name="ml_papers",
    persist_directory=chroma_dir,
    embedding_function=embeddings
)

if vectorstore._collection.count() == 0:
    vectorstore.add_documents(documents)
    print("Documentos indexados localmente en Chroma.")
else:
    print("Colección Chroma existente reutilizada; no se recalcularon embeddings.")

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1648.94it/s]


Colección Chroma existente reutilizada; no se recalcularon embeddings.


## Modelos de chat y cadenas para consulta de información

In [20]:
from langchain_openai import ChatOpenAI
from langchain_classic.chains import RetrievalQA

chat = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    model="gpt-5.6-luna",
    temperature=0
)

qa_chain = RetrievalQA.from_chain_type(
    llm=chat,
    chain_type="stuff",
    retriever=retriever
)

In [21]:
query = "qué es fingpt?"
qa_chain.run(query)

'FinGPT es un marco de código abierto para crear modelos de lenguaje especializados en finanzas (“FinLLMs”). Su objetivo es democratizar el acceso a datos financieros y facilitar el desarrollo de aplicaciones basadas en inteligencia artificial para este sector.\n\nSus principales características son:\n\n- **Recopilación de datos financieros**: noticias, redes sociales, informes regulatorios y otros datos de mercado.\n- **Enfoque centrado en los datos**: limpieza, curación y preprocesamiento de información financiera.\n- **Adaptación ligera del modelo**: técnicas de ajuste fino para mantenerlo actualizado con los cambios del mercado.\n- **Aplicaciones prácticas**: roboasesoría, análisis financiero, trading cuantitativo y desarrollo con poco código.\n- **Arquitectura de extremo a extremo**: cubre desde las fuentes de datos hasta las aplicaciones finales.\n\nEn resumen, FinGPT busca ofrecer una alternativa abierta y actualizable a los modelos financieros propietarios.'

In [ ]:
query = "qué hace complicado entrenar un modelo como el fingpt?"
qa_chain.run(query)

In [ ]:
query = "qué es fast segment?"
qa_chain.run(query)

In [ ]:
query = "cuál es la diferencia entre fast sam y mobile sam?"
qa_chain.run(query)